# Stanford Question Answering Dataset (SQuAD)

`rajpurkar/squad`

- Splits pulled: `train`, `validation`
- Output CSV: `data/squad_combined.csv`

```json
{
    "answers": {
        "answer_start": [1],
        "text": ["This is a test text"]
    },
    "context": "This is a test context.",
    "id": "1",
    "question": "Is this a test?",
    "title": "train test"
}
```


In [28]:
from csv import DictReader, DictWriter
from itertools import islice
from pathlib import Path

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

COLUMNS = [
    "split",
    "id",
    "title",
    "question",
    "context",
    "answer_text",
    "answer_start",
]

def load_split_records(split_name: str):
	filename = f"plain_text/{split_name}-00000-of-00001.parquet"
	local_path = hf_hub_download(
		repo_id="rajpurkar/squad",
		filename=filename,
		repo_type="dataset",
	)
	table = pq.read_table(local_path, use_pandas_metadata=False)
	data_dict = table.to_pydict()
	row_count = len(data_dict["id"])

	for idx in range(row_count):
		answers = data_dict["answers"][idx] or {}
		texts = answers.get("text") or [None]
		starts = answers.get("answer_start") or [None]

		for text, start in zip(texts, starts):
			yield {
				"split": split_name,
				"id": data_dict["id"][idx],
				"title": data_dict["title"][idx],
				"question": data_dict["question"][idx],
				"context": data_dict["context"][idx],
				"answer_text": text,
				"answer_start": start,
			}

def write_squad_csvs(output_dir: Path) -> dict:
	output_dir.mkdir(parents=True, exist_ok=True)
	results = {}
	for split_name in ("train", "validation"):
		out_path = output_dir / f"squad_{split_name}.csv"
		row_count = 0
		with out_path.open("w", newline="", encoding="utf-8") as fp:
			writer = DictWriter(fp, fieldnames=COLUMNS)
			writer.writeheader()
			for record in load_split_records(split_name):
				writer.writerow(record)
				row_count += 1
		results[split_name] = (out_path, row_count)
	return results

output_dir = Path(".") / "data"
results = write_squad_csvs(output_dir)

for split, (path, count) in results.items():
	print(f"Saved {split} CSV to: {path.resolve()} ({count:,} rows)")


Saved train CSV to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu8c/code/Users/manhductranvu/reeval-multi/custom-eval/data/squad_train.csv (87,599 rows)
Saved validation CSV to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu8c/code/Users/manhductranvu/reeval-multi/custom-eval/data/squad_validation.csv (34,726 rows)


In [ ]:
import pandas as pd
df1 = pd.read_parquet("./data/train-00000-of-00001.parquet")
df2 = pd.read_parquet("./data/train-00000-of-00002.parquet")
df3 = pd.read_parquet("./data/train-00001-of-00002.parquet")

df_all = pd.concat([df1, df2, df3], ignore_index=True)
df_all.to_csv("./data/hotpot_train.csv", index=False)
	
df = pd.read_parquet("./data/validation-00000-of-00001.parquet")
df.to_csv("./data/hotpot_validation.csv", index=False)
